In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp05
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/mlfp05/ex_3.py ──
"""
# ════════════════════════════════════════════════════════════════════════
# MLFP05 — Exercise 3: Shared Utilities
# ════════════════════════════════════════════════════════════════════════
#
# Common data loading, windowing, training, visualisation, and experiment
# tracking utilities shared across all technique files in this exercise.
#
# This module is NOT meant to be run standalone — import it from the
# technique files (01_vanilla_rnn.py, 02_lstm.py, etc.).
# ════════════════════════════════════════════════════════════════════════
"""

import asyncio
import pickle
from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset


from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker, ModelVisualizer
from kailash_ml import ModelRegistry
from kailash_ml.types import MetricSpec

# ── Constants ───────────────────────────────────────────────────────────
REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "mlfp05" / "stocks"
OUTPUT_DIR = REPO_ROOT / "outputs" / "mlfp05" / "ex_3"

TICKERS = {
    "^STI": "Straits Times Index",
    "D05.SI": "DBS Group Holdings",
    "9988.HK": "Alibaba HK",
    "AAPL": "Apple",
    "005930.KS": "Samsung",
    "7203.T": "Toyota",
}

SEQ_LEN = 20  # 20-day lookback (4 trading weeks)
FORECAST_HORIZON = 5  # predict next 5 days
FEATURES = ["Close", "High", "Low", "Volume"]
# Indicator-augmented feature set for 06_technical_indicators.py. Every
# indicator is computed from PAST closes only (rolling/ewm windows), so no
# future information leaks into the features.
INDICATOR_FEATURES = FEATURES + ["RSI", "MACD_hist", "BB_pctB"]
HIDDEN_DIM = 64
EPOCHS = 15
LR = 1e-3
CLIP = 1.0
BATCH_SIZE = 64


def init_environment() -> torch.device:
    """Set up environment, seeds, device, and output directories."""
    setup_environment()
    torch.manual_seed(42)
    np.random.seed(42)
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    device = get_device()
    print(f"Using device: {device}")
    return device


# ── Data Loading ────────────────────────────────────────────────────────
def fetch_ticker(symbol: str) -> pl.DataFrame:
    """Download daily OHLCV bars from yfinance, return polars DataFrame."""
    import yfinance as yf

    df = yf.download(
        symbol, start="2010-01-01", end="2024-12-31", progress=False, auto_adjust=True
    )
    if df is None or len(df) == 0:
        raise RuntimeError(f"yfinance returned empty frame for {symbol}")
    df = df.copy()
    df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]
    return pl.from_pandas(df.reset_index())


def load_or_fetch(symbol: str) -> tuple[pl.DataFrame | None, str]:
    """Load from parquet cache, or download and cache."""
    cache = DATA_DIR / f"{symbol.replace('^', '').replace('.', '_')}.parquet"
    if cache.exists():
        return pl.read_parquet(cache), "cache"
    try:
        df = fetch_ticker(symbol)
        df.write_parquet(cache)
        return df, "yfinance"
    except Exception as exc:
        print(f"  {symbol} unavailable ({type(exc).__name__}: {exc})")
        return None, "failed"


def load_stock_data() -> tuple[dict[str, pl.DataFrame], str, pl.DataFrame]:
    """Load all tickers and return (stock_data, primary_symbol, primary_df)."""
    stock_data: dict[str, pl.DataFrame] = {}
    for symbol, name in TICKERS.items():
        df, source = load_or_fetch(symbol)
        if df is not None:
            stock_data[symbol] = df
            print(f"  {symbol} ({name}): {len(df)} days [{source}]")

    if "^STI" not in stock_data and "AAPL" not in stock_data:
        raise RuntimeError("Need at least ^STI or AAPL data to proceed")

    primary = "^STI" if "^STI" in stock_data else "AAPL"
    primary_df = stock_data[primary]
    print(
        f"\nPrimary: {primary} -- {len(primary_df)} days, "
        f"{primary_df['Date'].min()} -> {primary_df['Date'].max()}"
    )
    return stock_data, primary, primary_df


# ── Technical Indicators (polars-native) ─────────────────────────────
def add_technical_indicators(df: pl.DataFrame) -> pl.DataFrame:
    """Append RSI(14), MACD(12,26,9) and Bollinger %B(20,2) in polars.

    All three are the canonical definitions:
      RSI:   Wilder's smoothed gain/loss ratio, 100 - 100/(1 + RS)
      MACD:  EMA(12) - EMA(26); signal = EMA(9) of MACD; hist = MACD - signal
      %B:    (Close - lower) / (upper - lower) with 20-day mean +/- 2 sd

    Every expression uses backward-looking rolling/ewm windows only, so the
    indicator on day t is computable at the close of day t — no lookahead.
    The first ~26 rows carry nulls (window warm-up); callers should
    ``drop_nulls()`` before building windowed datasets.
    """
    return df.with_columns(
        # ── RSI(14), Wilder's smoothing (ewm alpha = 1/14) ──────────
        pl.col("Close").diff().alias("_delta"),
    ).with_columns(
        pl.when(pl.col("_delta") > 0)
        .then(pl.col("_delta"))
        .otherwise(0.0)
        .alias("_gain"),
        pl.when(pl.col("_delta") < 0)
        .then(-pl.col("_delta"))
        .otherwise(0.0)
        .alias("_loss"),
    ).with_columns(
        pl.col("_gain").ewm_mean(alpha=1 / 14, adjust=False).alias("_avg_gain"),
        pl.col("_loss").ewm_mean(alpha=1 / 14, adjust=False).alias("_avg_loss"),
    ).with_columns(
        (
            100.0
            - 100.0 / (1.0 + pl.col("_avg_gain") / (pl.col("_avg_loss") + 1e-12))
        ).alias("RSI"),
        # ── MACD(12, 26, 9) ─────────────────────────────────────────
        (
            pl.col("Close").ewm_mean(span=12, adjust=False)
            - pl.col("Close").ewm_mean(span=26, adjust=False)
        ).alias("_macd"),
    ).with_columns(
        pl.col("_macd").ewm_mean(span=9, adjust=False).alias("_macd_signal"),
        # ── Bollinger %B(20, 2) ─────────────────────────────────────
        pl.col("Close").rolling_mean(20).alias("_bb_mid"),
        pl.col("Close").rolling_std(20).alias("_bb_std"),
    ).with_columns(
        (pl.col("_macd") - pl.col("_macd_signal")).alias("MACD_hist"),
        (
            (
                pl.col("Close")
                - (pl.col("_bb_mid") - 2.0 * pl.col("_bb_std"))
            )
            / (4.0 * pl.col("_bb_std") + 1e-12)
        ).alias("BB_pctB"),
    ).drop("_delta", "_gain", "_loss", "_avg_gain", "_avg_loss",
           "_macd", "_macd_signal", "_bb_mid", "_bb_std")


# ── Text Corpus (character-level modelling) ───────────────────────────
TEXT_DIR = REPO_ROOT / "data" / "mlfp05" / "text"
TINY_SHAKESPEARE_URL = (
    "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/"
    "tinyshakespeare/input.txt"
)


def load_text_corpus() -> tuple[str, str]:
    """Load a character-level text corpus, cached locally.

    Primary source: tiny-shakespeare (~1.1 MB of Shakespeare dialogue, the
    canonical char-RNN corpus), downloaded once and cached under
    ``data/mlfp05/text/``. If the download is unavailable (offline lab),
    the module's own textbook is used — the model then learns to write
    course-style prose, which is itself a fun demonstration.

    Returns: (text, corpus_name)
    """
    TEXT_DIR.mkdir(parents=True, exist_ok=True)
    cache = TEXT_DIR / "tiny_shakespeare.txt"
    if cache.exists():
        return cache.read_text(encoding="utf-8"), "tiny-shakespeare (cached)"

    import urllib.request

    try:
        with urllib.request.urlopen(TINY_SHAKESPEARE_URL, timeout=30) as resp:
            text = resp.read().decode("utf-8")
        cache.write_text(text, encoding="utf-8")
        return text, "tiny-shakespeare (downloaded)"
    except Exception as exc:
        textbook = REPO_ROOT / "modules" / "mlfp05" / "textbook.md"
        if textbook.exists():
            print(
                f"  tiny-shakespeare unavailable ({type(exc).__name__}); "
                "using the module textbook as corpus"
            )
            return textbook.read_text(encoding="utf-8"), "mlfp05 textbook"
        raise RuntimeError(
            f"No text corpus available: download failed ({exc}) and no "
            f"local textbook at {textbook}"
        ) from exc


# ── Windowed Datasets ───────────────────────────────────────────────────
def build_dataset(
    df: pl.DataFrame,
    seq_len: int = SEQ_LEN,
    horizon: int = FORECAST_HORIZON,
    feature_cols: list[str] | None = None,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, int]:
    """Build (seq_len window) -> (next horizon closes) arrays with z-score normalisation.

    Args:
        df: OHLCV frame (optionally indicator-augmented)
        seq_len: lookback window length
        horizon: forecast horizon
        feature_cols: columns to use as features (default: FEATURES).
            The TARGET is always the first column of feature_cols and must
            be "Close" — normalisation stats for the target come from the
            train split only.

    Returns: X, y, mean, std, n_train_windows
    """
    cols = FEATURES if feature_cols is None else feature_cols
    if cols[0] != "Close":
        raise ValueError(
            f"feature_cols[0] must be 'Close' (the forecast target); got {cols[0]!r}"
        )
    data = df.select(cols).to_numpy().astype(np.float32)
    n = len(data)
    split_n = int(0.8 * n)
    train_data = data[:split_n]
    mean = train_data.mean(axis=0, keepdims=True)
    std = train_data.std(axis=0, keepdims=True) + 1e-8
    data_norm = (data - mean) / std

    n_windows = n - seq_len - horizon + 1
    X = np.stack([data_norm[i : i + seq_len] for i in range(n_windows)])
    y = np.stack(
        [data_norm[i + seq_len : i + seq_len + horizon, 0] for i in range(n_windows)]
    )
    split_idx = split_n - seq_len
    return X.astype(np.float32), y.astype(np.float32), mean, std, split_idx


def prepare_dataloaders(
    primary_df: pl.DataFrame,
    device: torch.device,
    feature_cols: list[str] | None = None,
) -> tuple[
    DataLoader,
    DataLoader,
    torch.Tensor,
    torch.Tensor,
    torch.Tensor,
    torch.Tensor,
    np.ndarray,
    np.ndarray,
    int,
    int,
]:
    """Build train/val dataloaders and return raw tensors plus normalisation stats.

    Returns: train_loader, val_loader, X_train_t, y_train_t, X_val_t, y_val_t,
             norm_mean, norm_std, n_train_w, n_features
    """
    X_all, y_all, norm_mean, norm_std, n_train_w = build_dataset(
        primary_df, feature_cols=feature_cols
    )
    print(
        f"Built {len(X_all)} windows (seq_len={SEQ_LEN}, horizon={FORECAST_HORIZON}); "
        f"train {n_train_w}, val {len(X_all) - n_train_w}"
    )

    X_train_t = torch.from_numpy(X_all[:n_train_w]).to(device)
    y_train_t = torch.from_numpy(y_all[:n_train_w]).to(device)
    X_val_t = torch.from_numpy(X_all[n_train_w:]).to(device)
    y_val_t = torch.from_numpy(y_all[n_train_w:]).to(device)
    print(f"  X_train: {tuple(X_train_t.shape)}  y_train: {tuple(y_train_t.shape)}")
    print(f"  X_val:   {tuple(X_val_t.shape)}    y_val:   {tuple(y_val_t.shape)}")

    train_loader = DataLoader(
        TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True
    )
    val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=BATCH_SIZE)
    n_features = X_train_t.shape[-1]

    return (
        train_loader,
        val_loader,
        X_train_t,
        y_train_t,
        X_val_t,
        y_val_t,
        norm_mean,
        norm_std,
        n_train_w,
        n_features,
    )


# ── Experiment Tracking ─────────────────────────────────────────────────
async def _setup_engines(
    primary: str,
    experiment_suffix: str = "",
) -> tuple[ConnectionManager, ExperimentTracker, str, ModelRegistry | None, bool]:
    """Create ExperimentTracker (kailash-ml 1.1.1 factory) and ModelRegistry."""
    # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker
    # and ModelRegistry use incompatible _kml_model_versions schemas.
    # Route them to separate sqlite files until upstream fixes the conflict.
    db = "sqlite:///mlfp05_rnns.db"
    registry_db = "sqlite:///mlfp05_rnns_registry.db"
    tracker = await ExperimentTracker.create(store_url=db)
    conn = ConnectionManager(registry_db)
    await conn.initialize()
    registry = ModelRegistry(conn)
    exp_name = (
        f"m5_rnns_{primary}_{experiment_suffix}"
        if experiment_suffix
        else f"m5_rnns_{primary}"
    )
    return conn, tracker, exp_name, registry, True


def setup_engines(
    primary: str,
    experiment_suffix: str = "",
) -> tuple[ConnectionManager, ExperimentTracker, str, ModelRegistry | None, bool]:
    """Sync wrapper for engine setup."""
    return asyncio.run(_setup_engines(primary, experiment_suffix))


# ── Training Harness ────────────────────────────────────────────────────
def compute_gradient_norm(model: nn.Module) -> float:
    """Compute the total L2 norm of all gradients (before clipping)."""
    total_norm = 0.0
    for p in model.parameters():
        if p.grad is not None:
            total_norm += p.grad.data.norm(2).item() ** 2
    return total_norm**0.5


def _predict(model: nn.Module, x: torch.Tensor, attn: bool = False) -> torch.Tensor:
    """Forward pass, handling attention models that return a tuple."""
    out = model(x)
    return out[0] if attn else out


async def _train_model_async(
    model: nn.Module,
    name: str,
    tracker: ExperimentTracker,
    exp_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    device: torch.device,
    epochs: int = EPOCHS,
    lr: float = LR,
    clip: float = CLIP,
    attn: bool = False,
) -> dict[str, Any]:
    """Train with gradient tracking, log to ExperimentTracker."""
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    train_losses, val_losses, gradient_norms = [], [], []
    n_params = sum(p.numel() for p in model.parameters())

    async with tracker.track(experiment=exp_name, run_name=name) as run:
        await run.log_params(
            {
                "model_type": name,
                "hidden_dim": str(HIDDEN_DIM),
                "seq_len": str(SEQ_LEN),
                "forecast_horizon": str(FORECAST_HORIZON),
                "epochs": str(epochs),
                "lr": str(lr),
                "clip_norm": str(clip),
                "n_params": str(n_params),
            }
        )
        print(f"  [{name}] {n_params:,} parameters")

        for epoch in range(epochs):
            model.train()
            b_losses, e_grads = [], []
            for xb, yb in train_loader:
                opt.zero_grad()
                loss = F.mse_loss(_predict(model, xb, attn), yb)
                loss.backward()
                e_grads.append(compute_gradient_norm(model))
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip)
                opt.step()
                b_losses.append(loss.item())

            tl, gn = float(np.mean(b_losses)), float(np.mean(e_grads))
            train_losses.append(tl)
            gradient_norms.append(gn)

            model.eval()
            with torch.no_grad():
                vl = float(
                    np.mean(
                        [
                            F.mse_loss(_predict(model, xb, attn), yb).item()
                            for xb, yb in val_loader
                        ]
                    )
                )
            val_losses.append(vl)

            await run.log_metrics(
                {"train_loss": tl, "val_loss": vl, "gradient_norm": gn},
                step=epoch + 1,
            )
            print(
                f"  [{name}] epoch {epoch+1:2d}/{epochs}  "
                f"train={tl:.4f}  val={vl:.4f}  grad={gn:.4f}"
            )

        await run.log_metric("final_val_loss", val_losses[-1])

    return {
        "train_losses": train_losses,
        "val_losses": val_losses,
        "gradient_norms": gradient_norms,
        "final_val_loss": val_losses[-1],
    }


def train_model(
    model: nn.Module,
    name: str,
    tracker: ExperimentTracker,
    exp_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    device: torch.device,
    epochs: int = EPOCHS,
    lr: float = LR,
    clip: float = CLIP,
    attn: bool = False,
) -> dict[str, Any]:
    """Sync wrapper for training."""
    return asyncio.run(
        _train_model_async(
            model,
            name,
            tracker,
            exp_name,
            train_loader,
            val_loader,
            device,
            epochs,
            lr,
            clip,
            attn,
        )
    )


# ── Model Registry ──────────────────────────────────────────────────────
def register_best_model(
    model: nn.Module,
    model_name: str,
    val_loss: float,
    primary: str,
    registry: ModelRegistry | None,
    has_registry: bool,
) -> None:
    """Register a model in the ModelRegistry."""
    if not has_registry or registry is None:
        print("  ModelRegistry not available, skipping registration")
        return

    model_bytes = pickle.dumps(model.state_dict())
    # Architecture/ticker are encoded in the model name; numeric facts go to MetricSpec.
    reg_result = asyncio.run(
        registry.register_model(
            name=f"m5_rnn_{model_name.lower()}_{primary.replace('^', '')}",
            artifact=model_bytes,
            metrics=[
                MetricSpec(
                    name="val_loss", value=float(val_loss), higher_is_better=False
                ),
                MetricSpec(name="hidden_dim", value=float(HIDDEN_DIM)),
                MetricSpec(name="seq_len", value=float(SEQ_LEN)),
                MetricSpec(name="forecast_horizon", value=float(FORECAST_HORIZON)),
                MetricSpec(name="epochs", value=float(EPOCHS)),
            ],
        )
    )
    print(f"  Registered: {reg_result.name} v{reg_result.version}")


# ── Visualisation Helpers ───────────────────────────────────────────────
def get_visualizer() -> ModelVisualizer:
    """Create a ModelVisualizer instance.

    ModelVisualizer carries a kailash-ml P2 experimental notice (a
    UserWarning at construction); exercises run under warnings-as-errors,
    so the notice is acknowledged narrowly at this construction site.
    """
    import warnings as _warnings

    from kailash_ml._decorators import ExperimentalWarning as _EW

    with _warnings.catch_warnings():
        _warnings.simplefilter("ignore", _EW)
        return ModelVisualizer()


def plot_training_curves(
    viz: ModelVisualizer,
    results: dict[str, Any],
    model_name: str,
    output_prefix: str,
) -> None:
    """Plot training/validation loss curves and gradient norms."""
    train_metrics = {
        f"{model_name} train": results["train_losses"],
        f"{model_name} val": results["val_losses"],
    }
    viz.training_history(
        metrics=train_metrics, x_label="Epoch", y_label="MSE Loss"
    ).write_html(str(OUTPUT_DIR / f"{output_prefix}_training_curves.html"))

    grad_metrics = {model_name: results["gradient_norms"]}
    viz.training_history(
        metrics=grad_metrics, x_label="Epoch", y_label="Gradient L2 Norm"
    ).write_html(str(OUTPUT_DIR / f"{output_prefix}_gradient_norms.html"))


def plot_predictions(
    viz: ModelVisualizer,
    model: nn.Module,
    X_val_t: torch.Tensor,
    y_val_t: torch.Tensor,
    norm_mean: np.ndarray,
    norm_std: np.ndarray,
    output_prefix: str,
    attn: bool = False,
) -> tuple[np.ndarray, np.ndarray, torch.Tensor | None]:
    """Generate prediction-vs-actual scatter and return denormalised arrays.

    Returns: preds_denorm, actual_denorm, attn_weights (or None)
    """
    model.eval()
    with torch.no_grad():
        if attn:
            val_preds, attn_weights = model(X_val_t)
        else:
            val_preds = model(X_val_t)
            attn_weights = None

    close_mean, close_std = norm_mean[0, 0], norm_std[0, 0]
    preds_denorm = val_preds.cpu().numpy() * close_std + close_mean
    actual_denorm = y_val_t.cpu().numpy() * close_std + close_mean

    pred_df = pl.DataFrame(
        {
            "actual": actual_denorm[:, 0].tolist(),
            "predicted": preds_denorm[:, 0].tolist(),
        }
    )
    viz.scatter(pred_df, x="actual", y="predicted").write_html(
        str(OUTPUT_DIR / f"{output_prefix}_pred_vs_actual.html")
    )

    return preds_denorm, actual_denorm, attn_weights


def plot_time_series_overlay(
    preds_denorm: np.ndarray,
    actual_denorm: np.ndarray,
    output_prefix: str,
    title: str = "Predicted vs Actual Close Price",
    n_points: int = 200,
) -> None:
    """Plot predicted vs actual as overlaid time-series lines."""
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    n = min(n_points, len(preds_denorm))
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(
        range(n), actual_denorm[:n, 0], label="Actual", color="#2196F3", linewidth=1.5
    )
    ax.plot(
        range(n),
        preds_denorm[:n, 0],
        label="Predicted",
        color="#FF5722",
        linewidth=1.5,
        linestyle="--",
        alpha=0.85,
    )
    ax.set_xlabel("Validation Window Index")
    ax.set_ylabel("Close Price")
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(str(OUTPUT_DIR / f"{output_prefix}_time_series_overlay.png"), dpi=150)
    plt.close(fig)
    print(f"  Saved: {output_prefix}_time_series_overlay.png")


def plot_horizon_error(
    preds_denorm: np.ndarray,
    actual_denorm: np.ndarray,
    model_name: str,
) -> list[float]:
    """Print and return RMSE by forecast horizon day."""
    print(f"\n  Forecast Error by Horizon Day ({model_name}):")
    rmses = []
    for day in range(FORECAST_HORIZON):
        rmse = (
            float(np.mean((preds_denorm[:, day] - actual_denorm[:, day]) ** 2)) ** 0.5
        )
        rmses.append(rmse)
        print(f"    Day {day + 1}: RMSE={rmse:.2f}")
    return rmses




# ════════════════════════════════════════════════════════════════════════
# MLFP05 Exercise 3.6 — Technical Indicators as Features: RSI, MACD,
# Bollinger Bands (polars-native feature engineering)
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   After completing this file, you will be able to:
#   - Explain what RSI, MACD and Bollinger %B each measure — momentum,
#     trend divergence, and volatility position — in plain language
#   - Compute all three in POLARS with backward-looking windows only
#     (no lookahead leakage)
#   - Explain WHY indicators help sequence models: prices are
#     non-stationary, indicators are bounded or mean-reverting transforms
#   - Add indicator features to the windowed dataset and measure the
#     effect on forecast error honestly (it can help OR hurt)
#   - Evaluate a forecaster the way a trading desk does: directional
#     accuracy, not just RMSE
#   - Apply to an index-futures monitoring desk at a regional bank
#
# PREREQUISITES: M5/ex_3/02_lstm.py (LSTMRegressor, windowed datasets,
#   ExperimentTracker logging)
# ESTIMATED TIME: ~30 min
#
# DATASET: STI + APAC/global stocks via yfinance (2010-2024, parquet
#   cache). Indicators are computed from PAST closes only.
#
# PHASES:
#   1. THEORY  — What each indicator measures; stationarity; leakage
#   2. BUILD   — add_technical_indicators in polars; correctness checks
#   3. TRAIN   — LSTM on raw OHLCV vs indicator-augmented features
#   4. VISUALISE — Indicator panels + forecast overlays + horizon RMSE
#   5. APPLY   — Directional accuracy for an index-futures desk
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
import polars as pl
import torch
import torch.nn as nn

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt


device = init_environment()



## PHASE 1 — THEORY: Why Indicators, and Why They Help Sequence Models


RSI(14)      momentum        bounded [0, 100]   "gains vs losses"
  MACD hist    trend divergence mean-zero         "fast vs slow trend"
  Bollinger %B volatility pos.  centred at 0.5    "where in the envelope"

  All three are BACKWARD-LOOKING transforms of the close — computable at
  the close of day t, safe from lookahead leakage. Prices drift; these
  transforms stay in a fixed range. That is why networks learn them
  faster than raw price levels.


In [ ]:
# A price series is NON-STATIONARY: its mean and variance drift. Neural
# networks struggle with inputs whose distribution shifts — the same
# z-score means something different in 2012 and 2022.
#
# Technical indicators are HAND-CRAFTED STATIONARY TRANSFORMS, each
# compressing a different aspect of recent history:
#
#   RSI (Relative Strength Index, 14 days) — MOMENTUM
#     "Are recent gains overpowering recent losses?" Bounded in [0, 100]:
#     average gain vs average loss over two weeks, Wilder-smoothed.
#     >70 is conventionally "overbought", <30 "oversold". For the model:
#     a bounded input whose meaning does not drift with the price level.
#
#   MACD histogram (12/26/9) — TREND DIVERGENCE
#     "Is the fast trend pulling away from the slow trend, or closing?"
#     EMA(12) minus EMA(26), minus its own 9-day signal line. Mean-zero,
#     sign = momentum direction, magnitude = divergence strength.
#
#   Bollinger %B (20 days, 2 sd) — VOLATILITY POSITION
#     "Where is the price inside its own recent envelope?" 0.5 = at the
#     20-day mean; >1 = above the upper band; <0 = below the lower band.
#     Bounded-ish around 0.5 regardless of whether the index is at
#     3,000 or 30,000.
#
# LEAKAGE DISCIPLINE: every indicator at day t must be computable at the
# close of day t. Rolling and exponentially-weighted windows look only
# BACKWARD, so they are safe. (A centred rolling window would peek at
# future days — a classic leakage bug.) Normalisation statistics are
# computed on the train split only (build_dataset already does this).
#
# HONEST EXPECTATION: indicators add information the raw OHLCV window
# already contains in principle. Whether they help a 15-epoch LSTM is an
# empirical question — we measure it and report the sign we get.

print("=" * 70)
print("  PHASE 1 — THEORY: momentum, divergence, volatility position")
print("=" * 70)
print(
)



## PHASE 2 — BUILD: indicators in polars, with a hand-check


In [ ]:
print("=" * 70)
print("  PHASE 2 — BUILD: add_technical_indicators (polars)")
print("=" * 70)

stock_data, primary, primary_df = load_stock_data()
df_ind = add_technical_indicators(primary_df).drop_nulls()
print(
    f"\n{primary}: {len(primary_df)} rows -> {len(df_ind)} after indicator "
    "warm-up (first 26 rows carry null windows)"
)
print(
    df_ind.select(
        ["Date", "Close", "RSI", "MACD_hist", "BB_pctB"]
    ).tail(5)
)



### Checkpoint 1: indicators are correct


In [ ]:
# (a) RSI is bounded by construction
rsi = df_ind["RSI"]
assert float(rsi.min()) >= 0.0 and float(rsi.max()) <= 100.0, (
    f"RSI out of bounds: [{float(rsi.min()):.2f}, {float(rsi.max()):.2f}]"
)

# (b) Hand-check RSI at one row with an explicit Wilder recursion in numpy.
# polars alignment: row 0 has a null diff -> gain=loss=0, and
# ewm_mean(alpha=1/14, adjust=False) seeds the recursion with that row:
#   avg_0 = 0.0 ;  avg_r = (1 - alpha) * avg_{r-1} + alpha * x_r
# where x_r (r >= 1) is the gain/loss of closes[r] - closes[r-1].
closes = primary_df["Close"].to_numpy()
deltas = np.diff(closes)
gains = np.maximum(deltas, 0.0)
losses = np.maximum(-deltas, 0.0)
alpha = 1 / 14
avg_gain = avg_loss = 0.0
CHECK_AT = 200  # arbitrary raw-frame row well past the warm-up
for r in range(1, CHECK_AT + 1):
    avg_gain = (1 - alpha) * avg_gain + alpha * gains[r - 1]
    avg_loss = (1 - alpha) * avg_loss + alpha * losses[r - 1]
hand_rsi = 100.0 - 100.0 / (1.0 + avg_gain / (avg_loss + 1e-12))
# polars RSI column is aligned with df_ind rows; row r of df_ind is row
# r + (len(primary_df) - len(df_ind)) of the raw frame.
_offset = len(primary_df) - len(df_ind)
polars_rsi = float(df_ind["RSI"][CHECK_AT - _offset])
assert abs(hand_rsi - polars_rsi) < 0.5, (
    f"RSI hand-check failed: numpy recursion {hand_rsi:.2f} vs polars "
    f"{polars_rsi:.2f} at row {CHECK_AT}"
)

# (c) %B identity: 0.5 means "at the 20-day mean" — the column's own
# median should sit near 0.5 for a real price series.
pctb_median = float(df_ind["BB_pctB"].median())
assert 0.2 < pctb_median < 0.8, f"%B median {pctb_median:.3f} far from 0.5"

print(f"\n  RSI bounds: [{float(rsi.min()):.1f}, {float(rsi.max()):.1f}] (must be [0, 100])")
print(f"  RSI hand-check at row {CHECK_AT}: numpy={hand_rsi:.2f} polars={polars_rsi:.2f}")
print(f"  %B median: {pctb_median:.3f} (0.5 = at the 20-day mean)")
print("\n--- Checkpoint 1 passed --- indicators verified\n")



## PHASE 3 — TRAIN: raw OHLCV vs indicator-augmented LSTM


Single-layer LSTM forecaster (same architecture for both variants).


In [ ]:
print("=" * 70)
print("  PHASE 3 — TRAIN: same LSTM, same rows, different feature sets")
print("=" * 70)


class LSTMRegressor(nn.Module):

    def __init__(self, input_dim: int, hidden_dim: int = HIDDEN_DIM,
                 horizon: int = FORECAST_HORIZON):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.head = nn.Linear(hidden_dim, horizon)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out, _ = self.lstm(x)
        return self.head(out[:, -1])


conn, tracker, exp_name, registry, has_registry = setup_engines(
    primary, experiment_suffix="indicators"
)

# SAME trimmed frame for both variants — only the feature set differs.
(
    train_loader_base,
    val_loader_base,
    X_train_t,
    y_train_t,
    X_val_t_base,
    y_val_t_base,
    norm_mean,
    norm_std,
    n_train_w,
    n_features_base,
) = prepare_dataloaders(df_ind, device, feature_cols=FEATURES)

(
    train_loader_ind,
    val_loader_ind,
    _,
    _,
    X_val_t_ind,
    y_val_t_ind,
    _,
    _,
    _,
    n_features_ind,
) = prepare_dataloaders(df_ind, device, feature_cols=INDICATOR_FEATURES)

print(f"\nBaseline features ({n_features_base}): {FEATURES}")
print(f"Augmented features ({n_features_ind}): {INDICATOR_FEATURES}")

print("\nTraining baseline LSTM (OHLCV only)...")
torch.manual_seed(42)
model_base = LSTMRegressor(n_features_base).to(device)
res_base = train_model(
    model_base, "LSTM_OHLCV", tracker, exp_name,
    train_loader_base, val_loader_base, device, epochs=EPOCHS,
)

print("\nTraining indicator-augmented LSTM...")
torch.manual_seed(42)
model_ind = LSTMRegressor(n_features_ind).to(device)
res_ind = train_model(
    model_ind, "LSTM_OHLCV_indicators", tracker, exp_name,
    train_loader_ind, val_loader_ind, device, epochs=EPOCHS,
)



### Checkpoint 2: both models trained and converging


In [ ]:
for name, res in (("baseline", res_base), ("indicators", res_ind)):
    assert res["val_losses"][-1] < res["val_losses"][0], (
        f"{name}: val loss should decrease "
        f"({res['val_losses'][0]:.4f} -> {res['val_losses'][-1]:.4f})"
    )
    assert res["val_losses"][-1] < 0.5, (
        f"{name}: final val loss {res['val_losses'][-1]:.4f} too high — "
        "expected < 0.5 on normalised closes"
    )

delta = res_ind["final_val_loss"] - res_base["final_val_loss"]
print(f"\n{'=' * 62}")
print("  FEATURE-SET COMPARISON (measured, this run)")
print(f"{'=' * 62}")
print(f"  {'Variant':>28} {'Features':>9} {'Final Val MSE':>14}")
print("  " + "-" * 54)
print(f"  {'OHLCV baseline':>28} {n_features_base:>9} {res_base['final_val_loss']:>14.4f}")
print(f"  {'OHLCV + indicators':>28} {n_features_ind:>9} {res_ind['final_val_loss']:>14.4f}")
print(
    f"\n  Indicator delta: {delta:+.4f} MSE "
    f"({'helps' if delta < 0 else 'hurts'} at {EPOCHS} epochs). "
    "Either sign is a real measurement: indicators compress information "
    "the window already contains, so the benefit depends on horizon, "
    "architecture and training budget. What is NOT negotiable is the "
    "leakage discipline behind them."
)
print("\n--- Checkpoint 2 passed --- both variants trained\n")

best_is_ind = delta < 0
best_model = model_ind if best_is_ind else model_base
best_loss = res_ind["final_val_loss"] if best_is_ind else res_base["final_val_loss"]
register_best_model(
    best_model, "lstm_indicators" if best_is_ind else "lstm_ohlcv",
    best_loss, primary, registry, has_registry,
)



## PHASE 4 — VISUALISE: indicator panels + forecast quality


In [ ]:
print("=" * 70)
print("  PHASE 4 — VISUALISE: what the indicators see")
print("=" * 70)

# (a) Indicator panels over the last 500 trading days
panel = df_ind.tail(500)
days = np.arange(len(panel))
fig_ind, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
fig_ind.suptitle(
    f"{primary} — the three indicator views (last 500 trading days)", fontsize=14
)
axes[0].plot(days, panel["Close"].to_numpy(), color="#2196F3", linewidth=1.2)
axes[0].set_ylabel("Close")
axes[0].set_title("Price (the non-stationary raw series)")
axes[1].plot(days, panel["RSI"].to_numpy(), color="#9C27B0", linewidth=1.2)
axes[1].axhline(70, color="red", linestyle="--", alpha=0.5)
axes[1].axhline(30, color="green", linestyle="--", alpha=0.5)
axes[1].set_ylabel("RSI(14)")
axes[1].set_ylim(0, 100)
axes[1].set_title("Momentum — overbought > 70, oversold < 30")
macd_hist = panel["MACD_hist"].to_numpy()
axes[2].bar(days, macd_hist,
            color=np.where(macd_hist >= 0, "#4CAF50", "#F44336"), width=1.0)
axes[2].axhline(0, color="black", linewidth=0.5)
axes[2].set_ylabel("MACD hist")
axes[2].set_title("Trend divergence — fast EMA vs slow EMA, signal removed")
axes[3].plot(days, panel["BB_pctB"].to_numpy(), color="#FF9800", linewidth=1.2)
axes[3].axhline(1.0, color="red", linestyle="--", alpha=0.5)
axes[3].axhline(0.5, color="gray", linestyle=":", alpha=0.5)
axes[3].axhline(0.0, color="green", linestyle="--", alpha=0.5)
axes[3].set_ylabel("%B(20,2)")
axes[3].set_title("Volatility position — 0.5 = at the 20-day mean")
for ax in axes:
    ax.grid(True, alpha=0.3)
fig_ind.tight_layout()
fig_ind.savefig(str(OUTPUT_DIR / "06_indicator_panels.png"), dpi=150)
plt.close(fig_ind)
print(f"  Saved: {OUTPUT_DIR / '06_indicator_panels.png'}")

# (b) Training curves + forecast quality for both variants
# ModelVisualizer carries a kailash-ml P2 experimental notice (UserWarning
# at construction); the gate runs warnings-as-errors, so mute just that.
import warnings

from kailash_ml._decorators import ExperimentalWarning

with warnings.catch_warnings():
    warnings.simplefilter("ignore", ExperimentalWarning)
    viz = get_visualizer()
plot_training_curves(viz, res_base, "LSTM_OHLCV", "06_baseline")
plot_training_curves(viz, res_ind, "LSTM_indicators", "06_indicators")

preds_base, actual, _ = plot_predictions(
    viz, model_base, X_val_t_base, y_val_t_base, norm_mean, norm_std,
    "06_baseline",
)
preds_ind, _, _ = plot_predictions(
    viz, model_ind, X_val_t_ind, y_val_t_ind, norm_mean, norm_std,
    "06_indicators",
)
plot_time_series_overlay(
    preds_base, actual, "06_baseline",
    title=f"OHLCV baseline — predicted vs actual close ({primary})",
)
plot_time_series_overlay(
    preds_ind, actual, "06_indicators",
    title=f"Indicator-augmented — predicted vs actual close ({primary})",
)
rmse_base = plot_horizon_error(preds_base, actual, "OHLCV baseline")
rmse_ind = plot_horizon_error(preds_ind, actual, "OHLCV + indicators")



### Checkpoint 3: artefacts exist


In [ ]:
import os

for artefact in (
    OUTPUT_DIR / "06_indicator_panels.png",
    OUTPUT_DIR / "06_baseline_time_series_overlay.png",
    OUTPUT_DIR / "06_indicators_time_series_overlay.png",
):
    assert os.path.exists(artefact), f"Missing artefact: {artefact}"
print("\n--- Checkpoint 3 passed --- visual proof generated\n")



## PHASE 5 — APPLY: directional accuracy for an index-futures desk


Fraction of windows where predicted and realised 5-day moves agree in sign.


DIRECTIONAL ACCURACY ({len(actual_day5):,} validation windows, day-5 horizon):

    OHLCV baseline:       {hit_base:.1%}  (day-5 RMSE {rmse_base[-1]:.2f})
    OHLCV + indicators:   {hit_ind:.1%}  (day-5 RMSE {rmse_ind[-1]:.2f})
    coin-flip reference:  50.0%

  READING IT:
    A hit rate above 50% means the model carries real directional signal.
    Compare the RMSE ranking with the hit-rate ranking — when they
    disagree, the desk should trust the HIT RATE, because the hedge is a
    binary act (hedge / don't hedge), not a price-level bet.

  STAKEHOLDER-READY OUTPUT:
    "On {len(actual_day5):,} held-out windows the indicator-augmented
    forecaster calls the 5-day direction correctly {hit_ind:.0%} of the
    time vs {hit_base:.0%} for the raw-price baseline. Indicators are
    backward-looking transforms (no lookahead), computed in polars and
    audited against a hand-written RSI recursion."


In [ ]:
# SCENARIO (anonymised, illustrative): the treasury desk of a regional
# bank monitors index futures to hedge overnight exposure. The desk does
# not trade the model's price LEVEL — it acts on the model's DIRECTION
# call for the 5-day horizon: "will the index be higher or lower than
# the last close of the lookback window?"
#
# RMSE punishes magnitude errors; the desk's P&L cares about SIGN errors.
# A forecaster can halve its RMSE and still get the direction wrong on
# every trending day. So we score both variants on the metric the desk
# actually uses: directional hit rate = fraction of validation windows
# where the predicted 5-day move has the same sign as the realised move.

print("=" * 70)
print("  PHASE 5 — APPLY: directional accuracy (the desk's metric)")
print("=" * 70)

close_mean, close_std = norm_mean[0, 0], norm_std[0, 0]
# Last observed close of each validation window (denormalised)
last_close_base = X_val_t_base[:, -1, 0].cpu().numpy() * close_std + close_mean
last_close_ind = X_val_t_ind[:, -1, 0].cpu().numpy() * close_std + close_mean
actual_day5 = actual[:, -1]  # realised close at horizon day 5


def directional_hit_rate(preds: np.ndarray, last_close: np.ndarray) -> float:
    pred_move = preds[:, -1] - last_close
    real_move = actual_day5 - last_close
    agree = np.sign(pred_move) == np.sign(real_move)
    return float(np.mean(agree))


hit_base = directional_hit_rate(preds_base, last_close_base)
hit_ind = directional_hit_rate(preds_ind, last_close_ind)

print(
)



### Checkpoint 4: directional metrics computed honestly


In [ ]:
assert 0.0 <= hit_base <= 1.0 and 0.0 <= hit_ind <= 1.0
assert len(actual_day5) > 100, "Directional metric needs a real sample"
print("--- Checkpoint 4 passed --- desk-metric application demonstrated\n")



## REFLECTION


THEORY:
  [x] RSI = momentum (bounded [0,100]); MACD hist = trend divergence
      (mean-zero); Bollinger %B = volatility position (centred at 0.5)
  [x] Prices are non-stationary; indicators are stationary transforms —
      that is why networks learn them faster
  [x] Leakage discipline: backward-looking windows only, train-split
      normalisation only

  BUILD + TRAIN:
  [x] add_technical_indicators in pure polars, hand-verified: numpy
      Wilder recursion {hand_rsi:.2f} vs polars {polars_rsi:.2f}
  [x] Same LSTM, same rows, two feature sets ({n_features_base} vs
      {n_features_ind} columns), ExperimentTracker receipts
  [x] Measured effect: indicator delta {delta:+.4f} val MSE at
      {EPOCHS} epochs ({'helped' if delta < 0 else 'hurt'} this run —
      either sign is honest)

  VISUALISE (the proof):
  [x] Four-panel indicator chart: price / RSI / MACD hist / %B
  [x] Forecast overlays and horizon RMSE for both variants
      (day-5: {rmse_base[-1]:.2f} vs {rmse_ind[-1]:.2f})

  APPLY:
  [x] Directional accuracy — the metric a hedging desk is paid on
  [x] Baseline {hit_base:.1%} vs indicators {hit_ind:.1%} vs coin-flip 50%
  [x] When RMSE and hit-rate disagree, the binary desk action follows
      the hit rate

  KEY INSIGHT: Feature engineering is not "more columns = better model".
  It is the craft of turning domain knowledge into STATIONARY,
  LEAKAGE-FREE transforms — and then letting a measured ablation, not
  folklore, decide whether they stay. The audit trail matters as much
  as the delta: every indicator here is defined, hand-checked, and
  tracked in ExperimentTracker.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

